# Inclusive jet pseudorapidity by momentum correction

Select AK4 PF TightLepVeto jets in each stage's own half-open `[low, high)` pT
interval and project onto unflipped detector pseudorapidity. Trigger names and
intervals follow `06_data_jet_distributions.ipynb`. Use the original merged
jet-ID outputs so a cached nominal histogram cannot substitute for a different
residual campaign. The producer's no-residual mode is L2Relative only; it
does not load an L3Absolute payload.

The reference notebook's orientation overlays use unit-area normalization.
Here the primary curves retain the stored yields, as in its corrected/raw
diagnostic, so migration is visible. The eta window follows its eta display
range; the ±1.9 window used for its **pT spectra** instead divides the central
and outer diagnostics here. The plotted counts and diagnostics exclude flow
bins. Combined files retain detector eta in both beam directions.

Ratio error bars use the delta method with covariance set to zero. This is a
display approximation for overlapping events and cannot support significance
claims. Zero denominators are undefined; zero-content points are suppressed
visually. The CSV diagnostics retain measured zero ratios when applicable.
Marginal histograms measure net yield changes only. Matched jet membership
would separate entries and exits; paired per-event counts or a common event
bootstrap would provide the covariance and account for multiple jets per event.


In [ ]:
from pathlib import Path
import csv
import os
import sys

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / "CMakeLists.txt").is_file() and (p / "hist_analysis").is_dir())
sys.path.insert(0, str(PROJECT_ROOT))

from hist_analysis.python.notebook_setup import load_root
ROOT = load_root(batch=True)

from hist_analysis.config.files import BASE_DIR
from hist_analysis.config.histograms import trigger_single_jet_bins
from hist_analysis.python.jet_corrections_comparison import (
    REFERENCE, diagnostics, load_corrections, plot_comparison, project_eta, ratio_to_l2,
)


## Trigger, acceptance, and output configuration

In [ ]:
TRIGGERS = ('MinimumBias', 'Jet60', 'Jet80', 'Jet100')
JET_PT_BINS = {
    'MinimumBias': [(40, 60), (60, 80), (80, 100), (100, 120), (120, 140)],
    'Jet60': [(60, 80), (80, 100), (100, 120), (120, 140), (140, 180)],
    'Jet80': [(80, 100), (100, 120), (120, 140), (140, 180), (200, 300)],
    'Jet100': [(100, 120), (120, 140), (140, 180), (200, 300), (300, 500)],
}
SINGLE_JET_BINNING = "normal"  # "test" uses the shared test intervals
PT_BINS = trigger_single_jet_bins(JET_PT_BINS, SINGLE_JET_BINNING)
DIRECTIONS = ("combined", "Pbgoing", "pgoing")
ETA_RANGE = (-3.5, 3.5)
ETA_YLIM = None
RATIO_YLIM = None  # Set e.g. (0.8, 1.2) to zoom in on residual effects.
SAVE_PLOTS = True
OUTPUT_FORMATS = ("pdf", "png")
DATA_DIR = Path(os.environ.get("PPB_DATA_DIR", BASE_DIR / "exp"))
OUTPUT_DIR = Path(os.environ.get(
    "DATA_JET_JEC_OUTPUT_DIR",
    PROJECT_ROOT / "hist_analysis/output/data_jet_corrections_comparison",
)) / "jetId" / SINGLE_JET_BINNING


## Load and check input samples

Compare raw TH2 and selected-event vertex histogram contents, flow bins, entries,
and stored variances across campaigns. Matching aggregate controls cannot prove
event identity or authenticate the historical payloads. Omit the old residual
if its controls disagree, and print the affected trigger and direction.


In [ ]:
samples = {}
for trigger in TRIGGERS:
    for direction in DIRECTIONS:
        histograms, excluded = load_corrections(DATA_DIR, trigger, direction)
        samples[trigger, direction] = histograms, excluded
        for reason in excluded.values():
            print(f"{trigger}/{direction}: {reason}")


## Project and compare with L2-Rel.

In [ ]:
comparisons = {}
for (trigger, direction), (histograms, excluded) in samples.items():
    for pt in PT_BINS[trigger]:
        counts = {}
        for level, histogram in histograms.items():
            eta, values, variance = project_eta(histogram, pt, ETA_RANGE)
            counts[level] = values, variance
        ratios = {level: ratio_to_l2(pair, counts[REFERENCE])
                  for level, pair in counts.items() if level != REFERENCE}
        comparisons[trigger, direction, pt] = eta, counts, ratios, excluded


## Save standalone overlays and ratios

In [ ]:
figure_paths = []
if SAVE_PLOTS:
    for (trigger, direction, pt), (eta, counts, ratios, excluded) in comparisons.items():
        figure_paths.extend(plot_comparison(
            eta, counts, ratios, trigger=trigger, direction=direction, pt_range=pt,
            excluded=excluded, output_dir=OUTPUT_DIR, eta_ylim=ETA_YLIM,
            ratio_ylim=RATIO_YLIM, formats=OUTPUT_FORMATS,
        ))
print(f"Saved {len(figure_paths)} figure files to {OUTPUT_DIR}")


## Numerical diagnostics

Changes are fractions relative to L2-Rel. The largest bin deviation can
occur in a sparse bin; check its L2 yield before interpreting it. Central
and outer changes are signed yield shifts integrated over the corresponding
eta regions. Compare triggers descriptively: their offline intervals and
underlying selected samples are different.


In [ ]:
rows = []
for (trigger, direction, (low, high)), (eta, counts, ratios, excluded) in comparisons.items():
    for row in diagnostics(eta, counts, ratios):
        rows.append(dict(trigger=trigger, direction=direction, pt_low=low, pt_high=high,
                         eta_low=ETA_RANGE[0], eta_high=ETA_RANGE[1], **row))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
table_path = OUTPUT_DIR / f"jetId_etaLabUnflipped_{ETA_RANGE[0]:g}_{ETA_RANGE[1]:g}_yields.csv"
fields = list(dict.fromkeys(key for row in rows for key in row))
with table_path.open("w", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=fields)
    writer.writeheader()
    writer.writerows(rows)
print(f"Saved {len(rows)} yield rows to {table_path}")


# Retain every eta bin, including zero numerators and undefined ratios.
bins_path = table_path.with_name(table_path.name.replace("_yields", "_eta_bins"))
with bins_path.open("w", newline="") as stream:
    writer = csv.writer(stream)
    writer.writerow(("trigger", "direction", "pt_low", "pt_high", "level",
                     "eta_low", "eta_high", "yield", "sumw2",
                     "ratio_to_l2", "ratio_variance_zero_covariance"))
    for (trigger, direction, pt), (eta, counts, ratios, excluded) in comparisons.items():
        for level, (values, variances) in counts.items():
            for i, (value, variance) in enumerate(zip(values, variances)):
                ratio = (ratios[level][0][i], ratios[level][1][i]) if level in ratios else ("", "")
                writer.writerow((trigger, direction, *pt, level, eta[i], eta[i + 1],
                                 value, variance, *ratio))
print(f"Saved eta bins to {bins_path}")
